# 🧪 Unidad 2: Pipeline base de regresión

En este notebook se construirá un pipeline base para el problema de predicción del tiempo de permanencia de usuarios en una plataforma digital. El objetivo no es únicamente entrenar un modelo, sino comenzar a organizar el flujo de trabajo de una manera más cercana a un proyecto reproducible.


## Objetivos de aprendizaje

Al finalizar este notebook, el estudiante podrá:

- construir un conjunto de datos simulado para un problema de regresión;
- separar de manera explícita las etapas de datos, transformación y modelado;
- implementar un pipeline base con `scikit-learn`;
- evaluar un modelo de regresión utilizando métricas apropiadas;
- reconocer qué elementos del flujo todavía deben formalizarse para alcanzar un enfoque MLOps completo.


## 1. Punto de partida

En el notebook anterior se mostró por qué un notebook aislado no es suficiente para sostener un proyecto de Machine Learning. En esta sección avanzaremos un paso más: construiremos un pipeline base que permita hacer explícitas las etapas de preprocesamiento y entrenamiento.

Aún no estamos frente a un sistema MLOps completo, pero sí frente a una estructura mucho más clara, reutilizable y preparada para ser integrada posteriormente con herramientas como DVC y MLflow.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE: int = 42
rng = np.random.default_rng(RANDOM_STATE)


## 2. Función generadora de datos

Un primer paso hacia la modularidad consiste en encapsular la construcción de datos en funciones. En un proyecto real, esta función sería reemplazada por lectura desde archivos o bases de datos, pero aquí sirve para que el flujo permanezca completamente reproducible.


In [ ]:
def generate_session_dataset(n_samples: int, random_state: int = 42) -> pd.DataFrame:
    """Generate a mock dataset for session duration prediction.

    Parameters
    ----------
    n_samples : int
        Number of rows to generate.
    random_state : int, default=42
        Seed used for reproducibility.

    Returns
    -------
    pd.DataFrame
        Simulated dataset with predictors and target.
    """
    local_rng = np.random.default_rng(random_state)

    sites = np.array(["MLA", "MLB", "MLC", "MLM"])
    device_os = np.array(["android", "ios", "web"])
    segments = np.array(["new", "active", "at_risk", "churned"])
    entry_points = np.array(["home", "search", "recommendation", "push", "email"])

    data = pd.DataFrame({
        "user_id": np.arange(1, n_samples + 1),
        "site": local_rng.choice(sites, size=n_samples, p=[0.35, 0.20, 0.15, 0.30]),
        "device_os": local_rng.choice(device_os, size=n_samples, p=[0.55, 0.25, 0.20]),
        "segment": local_rng.choice(segments, size=n_samples, p=[0.20, 0.45, 0.20, 0.15]),
        "entry_point": local_rng.choice(entry_points, size=n_samples, p=[0.25, 0.25, 0.20, 0.20, 0.10]),
        "hour_of_day": local_rng.integers(0, 24, size=n_samples),
        "day_of_week": local_rng.integers(0, 7, size=n_samples),
        "historical_avg_session_minutes": local_rng.uniform(2.0, 35.0, size=n_samples),
        "historical_sessions_last_7d": local_rng.integers(1, 25, size=n_samples),
        "days_since_last_session": local_rng.integers(0, 30, size=n_samples),
        "push_received_last_24h": local_rng.integers(0, 6, size=n_samples),
    })

    segment_effect = data["segment"].map({
        "new": -1.5,
        "active": 3.5,
        "at_risk": -2.0,
        "churned": -4.0,
    })

    entry_effect = data["entry_point"].map({
        "home": 1.2,
        "search": 2.0,
        "recommendation": 3.0,
        "push": 1.5,
        "email": 0.8,
    })

    device_effect = data["device_os"].map({
        "android": 0.5,
        "ios": 0.8,
        "web": 1.5,
    })

    hour_effect = np.where(data["hour_of_day"].between(18, 22), 2.2, 0.0)
    weekend_effect = np.where(data["day_of_week"].isin([5, 6]), 1.3, 0.0)
    noise = local_rng.normal(loc=0.0, scale=2.5, size=n_samples)

    data["session_minutes"] = (
        1.5
        + 0.55 * data["historical_avg_session_minutes"]
        + 0.30 * data["historical_sessions_last_7d"]
        - 0.25 * data["days_since_last_session"]
        - 0.40 * data["push_received_last_24h"]
        + segment_effect
        + entry_effect
        + device_effect
        + hour_effect
        + weekend_effect
        + noise
    ).clip(lower=0.5)

    return data


In [ ]:
data = generate_session_dataset(n_samples=20000, random_state=RANDOM_STATE)
data.head()


## 3. Inspección inicial del dataset

Antes de modelar, es importante comprender la forma general del dataset, validar tipos de datos y tener una idea de la variable objetivo.


In [ ]:
data.info()


In [ ]:
data["session_minutes"].describe()


## 4. Definición explícita de entradas y salida

Un pipeline claro parte de una separación explícita entre la variable objetivo y el conjunto de predictores.


In [ ]:
target_column: str = "session_minutes"
feature_columns = [
    "site",
    "device_os",
    "segment",
    "entry_point",
    "hour_of_day",
    "day_of_week",
    "historical_avg_session_minutes",
    "historical_sessions_last_7d",
    "days_since_last_session",
    "push_received_last_24h",
]

X = data[feature_columns].copy()
y = data[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_test.shape)


## 5. Construcción del pipeline

Aquí comienza uno de los puntos clave del notebook: en lugar de transformar datos por fuera y luego entrenar un modelo de forma separada, se definirá un pipeline que conecte ambas etapas.

Esto tiene varias ventajas:

- disminuye errores de fuga de información;
- organiza el flujo en una sola unidad;
- facilita la reutilización del código;
- prepara el camino para su posterior integración en un proyecto más formal.


In [ ]:
categorical_features = ["site", "device_os", "segment", "entry_point"]
numeric_features = [
    "hour_of_day",
    "day_of_week",
    "historical_avg_session_minutes",
    "historical_sessions_last_7d",
    "days_since_last_session",
    "push_received_last_24h",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]
)

regression_pipeline


## 6. Entrenamiento y evaluación

Una vez definido el pipeline, el entrenamiento se vuelve más limpio: el preprocesamiento y el modelo quedan encapsulados dentro de una sola estructura.


In [ ]:
regression_pipeline.fit(X_train, y_train)
predictions = regression_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)
r2 = r2_score(y_test, predictions)

metrics_df = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae, rmse, r2],
})
metrics_df


## 7. Interpretación de resultados

Este resultado ya es más útil que el de un flujo desordenado, porque:

- el preprocesamiento es explícito;
- el modelo está encapsulado en un pipeline;
- el flujo puede repetirse con los mismos pasos.

Aun así, todavía faltan varios elementos para hablar realmente de un sistema MLOps:

- no hay versionamiento del dataset;
- no hay tracking de parámetros y métricas;
- no hay separación del pipeline en scripts reutilizables;
- no existe una estructura formal de proyecto;
- no hay control de entorno ni de colaboración.


## 8. De pipeline en notebook a pipeline en proyecto

Una siguiente evolución razonable sería mover este flujo a un proyecto organizado, por ejemplo:

```text
session-duration-mlops/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
│   └── 04_pipeline_base_regresion.ipynb
├── src/
│   ├── data/
│   │   └── make_dataset.py
│   ├── features/
│   │   └── build_features.py
│   ├── models/
│   │   └── train_model.py
│   └── evaluation/
│       └── evaluate_model.py
├── outputs/
├── models/
├── pyproject.toml
├── params.yaml
├── dvc.yaml
└── README.md
```

Observa que el notebook seguiría existiendo, pero ya no como centro del proyecto, sino como material de exploración y documentación.


## 9. Ejercicio guiado

A partir del pipeline construido, realiza lo siguiente:

1. Cambia al menos un hiperparámetro del modelo y observa qué ocurre con las métricas.
2. Identifica qué parte del notebook debería convertirse primero en un script reutilizable.
3. Propón qué elementos deberían ir en un archivo `params.yaml`.
4. Explica por qué este pipeline representa un avance frente al notebook anterior, pero aún no constituye un flujo MLOps completo.


## 10. Idea clave de cierre

El aprendizaje principal de este notebook es el siguiente:

> un pipeline bien definido organiza el flujo del modelo, pero solo se convierte en una solución MLOps cuando se integra con versionamiento, reproducibilidad, tracking y colaboración.

En las siguientes unidades comenzaremos a formalizar justamente esos componentes.
